# Co-training dan tri-training: semi-supervised dengan beberapa "sudut pandang" teks
**Kapan dipakai:** label sedikit, teks tanpa label banyak (test, atau file tambahan). Teks punya
dua sudut pandang alami yang cukup berdiri sendiri: **kata** (TF-IDF n-gram kata) dan **huruf**
(TF-IDF n-gram huruf). Bisa juga teks vs kolom tabular.

| cara | ide | sumber |
|---|---|---|
| hanya label | rata-rata model kata + huruf, tanpa data tak berlabel | patokan |
| self-training | satu model memberi label pada tebakan yakinnya sendiri | Yarowsky 1995 |
| co-training | model kata dan model huruf saling memberi label baru | Blum & Mitchell 1998 |
| tri-training | 3 model, baris ditambah ke model i bila 2 model lain sepakat | Zhou & Li 2005 |

Beda dengan `semi_supervised_teks` (pseudo-label satu model) dan `graf_label_teks` (graf
tetangga). Tri-training di sini versi sederhana: syarat batas galat dari makalah asli diganti
ambang keyakinan `TRI_THRESHOLD`.

**Skor jujur:** lipatan hanya atas baris BERLABEL. Baris validasi tidak pernah masuk kolam
tak berlabel.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

## Konfigurasi

In [ ]:
OUTPUT_FOLDER = "../../../outputs/text/co_training_teks"
DATA_FOLDER = "../../../data_latihan/text/classification"    # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/emot_train.csv"                  # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/emot_test.csv"                    # GANTI saat lomba
KEY_CSV = f"{DATA_FOLDER}/emot_test_labels.csv"              # GANTI saat lomba (kunci, kalau ada)
UNLABELED_CSV = None                         # GANTI: teks tambahan tanpa label (kolom TEXT_COLS)
ID_COL = "id"                                # GANTI saat lomba
TARGET = "label"                             # GANTI saat lomba
TEXT_COLS = ["text"]                         # GANTI saat lomba
LABEL_SHARE = 0.15           # demo: hanya 15% label train dipakai. Saat lomba: 1.0
SELF_ROUNDS = 5
SELF_THRESHOLD = 0.7
CO_ROUNDS = 8
CO_PER_CLASS = 10            # baris baru per kelas per model per putaran
TRI_ROUNDS = 4
TRI_THRESHOLD = 0.5
N_FOLDS = 5
SEED = 42


def metric(truth, pred):
    return f1_score(truth, pred.argmax(axis=1), average="macro")      # GANTI: metrik lomba


print("uji metrik:", metric(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]])))

## Baca data + dua sudut pandang (kata, huruf)

In [ ]:
raw_train = pd.read_csv(TRAIN_CSV)
train = raw_train[raw_train[TARGET].notna()].reset_index(drop=True)
test = pd.read_csv(TEST_CSV)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
tables = [train, test]
no_label = raw_train[raw_train[TARGET].isna()]       # target kosong → kolam tak berlabel
if len(no_label) > 0:
    tables.append(no_label)
if UNLABELED_CSV is not None:
    tables.append(pd.read_csv(UNLABELED_CSV))
texts = []
for table in tables:
    text = table[TEXT_COLS].fillna("")
    texts.append(text.astype(str).agg(" . ".join, axis=1))
all_text = pd.concat(texts, ignore_index=True)
CLASSES, y = np.unique(train[TARGET].astype(str), return_inverse=True)
word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000,
                       sublinear_tf=True).fit_transform(all_text)
char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=100000,
                       sublinear_tf=True).fit_transform(all_text)
both = hstack([word, char]).tocsr()
n_train = len(train)
n_test = len(test)
print("baris:", len(all_text), "| kata:", word.shape, "| huruf:", char.shape)

## Baris berlabel vs kolam tak berlabel
Kolam = train yang labelnya disembunyikan (demo) + test + baris train bertarget kosong +
`UNLABELED_CSV`.

In [ ]:
rng = np.random.default_rng(SEED)
known = rng.random(n_train) < LABEL_SHARE
labeled = np.flatnonzero(known)
unlabeled = np.r_[np.flatnonzero(~known), np.arange(n_train, len(all_text))]
y_all = np.full(len(all_text), -1)
y_all[:n_train] = y
inner = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(labeled, y[labeled])
folds = []
for idx_fit, idx_val in inner:
    folds.append((labeled[idx_fit], labeled[idx_val]))
print("berlabel:", len(labeled), "| kolam tak berlabel:", len(unlabeled))

## Alat bersama

In [ ]:
def fit_view(matrix, rows, labels):
    fitted = SGDClassifier(loss="log_loss", alpha=1e-5, random_state=SEED)
    fitted.fit(matrix[rows], labels)
    proba = np.zeros((matrix.shape[0], len(CLASSES)))
    proba[:, fitted.classes_] = fitted.predict_proba(matrix)
    return proba


def two_views(rows, labels):
    return (fit_view(word, rows, labels) + fit_view(char, rows, labels)) / 2


def top_confident(proba, pool):
    picked = [np.zeros(0, dtype=int)]
    guess = proba[pool].argmax(axis=1)
    for number in range(len(CLASSES)):
        members = pool[guess == number]
        order = np.argsort(-proba[members, number])
        picked.append(members[order[:CO_PER_CLASS]])
    return np.unique(np.concatenate(picked))


print("uji pilih:", top_confident(np.eye(len(CLASSES))[[0, 1, 1]], np.arange(3)))

## Cara 1–2: hanya label, self-training

In [ ]:
def supervised_run(idx_fit):
    return two_views(idx_fit, y_all[idx_fit])


def self_training_run(idx_fit):
    rows, labels = idx_fit, y_all[idx_fit]
    pool = unlabeled
    for _ in range(SELF_ROUNDS):
        proba = two_views(rows, labels)
        sure = pool[proba[pool].max(axis=1) >= SELF_THRESHOLD]
        if len(sure) == 0:
            break
        rows = np.r_[rows, sure]
        labels = np.r_[labels, proba[sure].argmax(axis=1)]
        pool = np.setdiff1d(pool, sure)
    return two_views(rows, labels)


print("fungsi patokan + self-training siap")

## Cara 3: co-training (kata ↔ huruf)
Tiap putaran: model kata memilih `CO_PER_CLASS` baris paling yakin per kelas dari kolam dan
menambahkannya ke himpunan berlabel bersama, lalu model huruf melakukan hal yang sama.

In [ ]:
def co_training_run(idx_fit):
    rows, labels = idx_fit, y_all[idx_fit]
    pool = unlabeled
    for _ in range(CO_ROUNDS):
        for matrix in [word, char]:
            proba = fit_view(matrix, rows, labels)
            picked = top_confident(proba, pool)
            rows = np.r_[rows, picked]
            labels = np.r_[labels, proba[picked].argmax(axis=1)]
            pool = np.setdiff1d(pool, picked)
    return two_views(rows, labels)


print("fungsi co-training siap")

## Cara 4: tri-training (kata, huruf, gabungan)

In [ ]:
TRI_VIEWS = [word, char, both]


def agreed_rows(probas, number):
    others = []
    for other in range(len(probas)):
        if other != number:
            others.append(probas[other][unlabeled])
    first, second = others
    first_guess = first.argmax(axis=1)
    same = first_guess == second.argmax(axis=1)
    sure = np.minimum(first.max(axis=1), second.max(axis=1)) >= TRI_THRESHOLD
    return unlabeled[same & sure], first_guess[same & sure]


demo_probas = [np.tile(np.eye(len(CLASSES))[0], (len(all_text), 1))] * 3
print("uji sepakat:", len(agreed_rows(demo_probas, 0)[0]) == len(unlabeled))

In [ ]:
def tri_training_run(idx_fit):
    labels_fit = y_all[idx_fit]
    probas = []
    for matrix in TRI_VIEWS:
        probas.append(fit_view(matrix, idx_fit, labels_fit))
    for _ in range(TRI_ROUNDS):
        updated = []
        for number, matrix in enumerate(TRI_VIEWS):
            extra, extra_labels = agreed_rows(probas, number)
            rows = np.r_[idx_fit, extra]
            updated.append(fit_view(matrix, rows, np.r_[labels_fit, extra_labels]))
        probas = updated
    return np.mean(probas, axis=0)


print("fungsi tri-training siap")

## Jalankan semua (lipatan sama, hanya baris berlabel yang dinilai)

In [ ]:
METHODS = {"hanya label": supervised_run, "self-training": self_training_run,
           "co-training": co_training_run, "tri-training": tri_training_run}
RESULTS = {}
rows = []
for name, runner in METHODS.items():
    start = time.time()
    oof = np.zeros((n_train, len(CLASSES)))
    test_pred = np.zeros((n_test, len(CLASSES)))
    for idx_fit, idx_val in folds:
        proba = runner(idx_fit)
        oof[idx_val] = proba[idx_val]
        test_pred += proba[n_train: n_train + n_test] / N_FOLDS
    RESULTS[name] = (oof, test_pred)
    rows.append({"cara": name, "OOF berlabel": round(metric(y[labeled], oof[labeled]), 4),
                 "detik": round(time.time() - start)})
    print(rows[-1])

## Ringkasan + gambar
Pilih cara dengan OOF berlabel terbaik. Kalau selisih dengan "hanya label" < 0,005, data tak
berlabel tidak membantu: pakai yang sederhana.

In [ ]:
table = pd.DataFrame(rows)
table.to_csv(f"{OUTPUT_FOLDER}/banding_cara.csv", index=False)
CHOSEN = table["cara"][int(table["OOF berlabel"].idxmax())]
figure, ax = plt.subplots(figsize=(7, 3.2))
ax.barh(table["cara"], table["OOF berlabel"], color="#0072B2")
ax.set_xlim(table["OOF berlabel"].min() - 0.03, table["OOF berlabel"].max() + 0.01)
ax.set_xlabel(f"F1 OOF (baris berlabel, {len(labeled)} baris)")
ax.invert_yaxis()
figure.savefig(f"{OUTPUT_FOLDER}/banding_cara.png", dpi=200, bbox_inches="tight")
plt.show()
guess = pd.DataFrame()
guess[ID_COL] = test[ID_COL]
guess[TARGET] = CLASSES[RESULTS[CHOSEN][1].argmax(axis=1)]
guess.to_csv(f"{OUTPUT_FOLDER}/tebakan_test.csv", index=False)
print("dipilih:", CHOSEN)

## Cek dengan kunci jawaban (HAPUS saat lomba)
Demo saja: kunci test dipakai untuk melihat apakah OOF berlabel memilih cara yang benar.

In [ ]:
key = pd.read_csv(KEY_CSV)
key_by_id = pd.Series(key[TARGET].astype(str).values, index=key[ID_COL].values)
truth_test = np.searchsorted(CLASSES, test[ID_COL].map(key_by_id).to_numpy())
for name in RESULTS:
    print(f"{name:14s} | test {metric(truth_test, RESULTS[name][1]):.4f}")